# chaotic-pfc Tour

This notebook walks through the complete pipeline: from the Hénon map
to FIR-filtered chaotic communication and Lyapunov parameter sweeps.

- Runtime: ~1 minute
- Installation: `pip install -e .` (editable mode from the repository root)

---

## Tour pelo chaotic-pfc

Este notebook percorre o pipeline completo: do mapa de Hénon à
comunicação caótica com filtro FIR e varreduras de parâmetros de Lyapunov.

- Duração: ~1 minuto
- Instalação: `pip install -e .` (modo editável a partir da raiz do repositório)


In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import firwin, welch

from chaotic_pfc.analysis.sweep import quick_sweep_params, run_sweep
from chaotic_pfc.analysis.sweep_plotting import (
    plot_classification_interleaved,
    plot_heatmap_continuous,
)
from chaotic_pfc.dynamics.lyapunov import lyapunov_henon2d, lyapunov_henon2d_ensemble
from chaotic_pfc.dynamics.maps import henon_filtered, henon_standard

print("Imports OK")

## 1. Mapa de Hénon

O mapa de Hénon é um sistema dinâmico discreto 2D. A biblioteca usa a forma generalizada:

$$ x_{n+1} = \\alpha - x_n^2 + \\beta \\, y_n, \\qquad y_{n+1} = x_n $$

equivalente à forma clássica sob mudança linear de variáveis.

Para $\\alpha = 1.4$, $\\beta = 0.3$, o sistema exibe caos determinístico.

---

## 1. Hénon Map

The Hénon map is a 2D discrete dynamical system. The library uses the generalised form:

$$ x_{n+1} = \\alpha - x_n^2 + \\beta \\, y_n, \\qquad y_{n+1} = x_n $$

equivalent to the classic form under a linear change of variables.

For $\\alpha = 1.4$, $\\beta = 0.3$, the system exhibits deterministic chaos.


In [ ]:
N_points = 3000
burn_in = 500

X, Y = henon_standard(N_points + burn_in, x0=0.0, y0=0.0)
x, y = X[burn_in:], Y[burn_in:]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(x, y, ",k", alpha=0.3, markersize=0.5)
ax1.set_xlabel("$x_n$")
ax1.set_ylabel("$y_n$")
ax1.set_title("Hénon Attractor")
ax2.plot(range(200), x[:200], linewidth=0.5)
ax2.set_xlabel("Iteration $n$")
ax2.set_ylabel("$x_n$")
ax2.set_title("Time series (200 pts)")
plt.tight_layout()
plt.show()

## 2. Expoentes de Lyapunov: cálculo direto

$\\lambda_{\\max} > 0$: caótico. $\\lambda_{\\max} \\le 0$: periódico/estável.

Wolf et al. (1985) reportam $\\lambda_1 \\approx 0.418$ para o mapa de Hénon
(forma generalizada equivalente). Validação: `docs/validation.rst`.

---

## 2. Lyapunov Exponents: direct computation

$\\lambda_{\\max} > 0$: chaotic. $\\lambda_{\\max} \\le 0$: periodic/stable.

Wolf et al. (1985) report $\\lambda_1 \\approx 0.418$ for the Hénon map
(generalised form, equivalent). Validation: `docs/validation.rst`.


In [ ]:
result = lyapunov_henon2d(alpha=1.4, beta=0.3, Nitera=2000, Ndiscard=1000, seed=42)
print(f"λ₁ = {result.all_exponents[0]:.6f}")
print(f"λ₂ = {result.all_exponents[1]:.6f}")
print(f"λ_max = {result.lyapunov_max:.6f}")
print(f"λ₁ + λ₂ = {np.sum(result.all_exponents):.6f}  (ln(0.3) = {np.log(0.3):.6f})")
print(f"Chaotic? {'Yes' if result.lyapunov_max > 0 else 'No'}")

## 3. Ensemble de Lyapunov

Múltiplas condições iniciais ao redor do ponto fixo para avaliar robustez.

---

## 3. Lyapunov Ensemble

Multiple initial conditions around the fixed point to assess robustness.


In [ ]:
ensemble = lyapunov_henon2d_ensemble(
    alpha=1.4,
    beta=0.3,
    Nitera=2000,
    Ndiscard=1000,
    n_initial=10,
    seed=42,
    perturbation=0.1,
)
n_total = len(ensemble.lmax_per_ci)
print(f"Mean λ_max: {ensemble.mean_lmax:.6f}")
print(
    f"Chaotic: {ensemble.n_chaotic}/{n_total}  Periodic: {n_total - ensemble.n_chaotic}/{n_total}"
)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(ensemble.lmax_per_ci, bins=8, edgecolor="k", alpha=0.7)
ax.axvline(0, color="gray", linestyle="--", label="Threshold")
ax.set_xlabel("$\\lambda_{\\max}$")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Mapa de Hénon filtrado (FIR)

**Motivação do PFC**: filtro FIR no laço de realimentação limita a banda
do sinal caótico, essencial para canais de comunicação reais.

---

## 4. FIR-Filtered Hénon Map

**PFC motivation**: FIR filter in the feedback loop band-limits the
chaotic signal for real communication channels.


In [ ]:
N_filter, wc = 5, 0.5
coeffs = firwin(N_filter + 1, wc)
print("FIR coefficients:")
print(f"  {np.array2string(coeffs, precision=4)}")
X_std, Y_std = henon_standard(3000, x0=0.0, y0=0.0)
X_filt, Y_filt = henon_filtered(3000, x0=0.0, y0=0.0, c0=coeffs[0], c1=coeffs[1])
x_std, x_filt, y_filt = X_std[500:], X_filt[500:], Y_filt[500:]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(x_filt, y_filt, ",k", alpha=0.3, markersize=0.5)
ax1.set_xlabel("$x_n$")
ax1.set_ylabel("$y_n$")
ax1.set_title(f"Filtered Hénon (FIR N={N_filter})")
f, pxx_std = welch(x_std, fs=1.0, nperseg=256)
f, pxx_filt = welch(x_filt, fs=1.0, nperseg=256)
ax2.semilogy(f, pxx_std, alpha=0.5, label="Standard")
ax2.semilogy(f, pxx_filt, alpha=0.5, label="Filtered")
ax2.set_xlabel("Normalised frequency")
ax2.set_ylabel("PSD")
ax2.legend()
plt.tight_layout()
plt.show()

## 4b. Comunicação caótica: transmissão e recepção

Modulamos uma mensagem binária (±1) no portador caótico via Pecora-Carroll.
O receptor sincroniza e recupera a mensagem.

---

## 4b. Chaotic Communication: transmission and reception

A binary message (±1) is modulated onto the chaotic carrier via Pecora-Carroll.
The receiver synchronises and recovers the message.


In [ ]:
from chaotic_pfc.comms.channel import awgn, ideal_channel
from chaotic_pfc.comms.receiver import receive
from chaotic_pfc.comms.transmitter import transmit

N_bits = 1000
msg = np.ones(N_bits)
msg[::20] = -1  # periodic message

# Transmit
s = transmit(msg, mu=0.01, a=1.4, b=0.3, x0=0.1, y0=0.1)

# Channel (ideal + AWGN)
r_clean = ideal_channel(s)
r_noisy = awgn(s, snr_db=20, rng=np.random.default_rng(42))

# Receive
m_clean = receive(r_clean, mu=0.01, a=1.4, b=0.3, y0=0.1, z0=0.1)
m_noisy = receive(r_noisy, mu=0.01, a=1.4, b=0.3, y0=0.1, z0=0.1)

# Skip transient and compute BER
skip = 200
err_clean = np.mean(np.sign(m_clean[skip:]) != np.sign(msg[skip:]))
err_noisy = np.mean(np.sign(m_noisy[skip:]) != np.sign(msg[skip:]))
print(f"BER (ideal):  {err_clean:.4f}")
print(f"BER (SNR=20): {err_noisy:.4f}")

# Plot
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(msg[:100], "k-", linewidth=0.8, label="Original")
axes[0].plot(m_clean[:100], "b--", linewidth=0.8, alpha=0.6, label="Recovered")
axes[0].set_ylabel("Message")
axes[0].legend(fontsize=8)
axes[0].set_title("Ideal channel")
axes[1].plot(msg[:100], "k-", linewidth=0.8, label="Original")
axes[1].plot(m_noisy[:100], "r--", linewidth=0.8, alpha=0.6, label="Recovered")
axes[1].set_xlabel("Sample $n$")
axes[1].set_ylabel("Message")
axes[1].legend(fontsize=8)
axes[1].set_title("AWGN channel (SNR=20 dB)")
plt.tight_layout()
plt.show()

## 5. Mini-sweep: varredura de parâmetros

Varredura da grade $(N_z, \\omega_c)$, classificação por ponto.

---

## 5. Mini-Sweep: parameter scan

Grid scan over $(N_z, \\omega_c)$, per-point classification.


In [ ]:
orders_lp, orders_hp, cutoffs, params = quick_sweep_params()
print(f"Grid: {len(orders_lp)}×{len(cutoffs)} = {len(orders_lp) * len(cutoffs)} points")
t0 = time.time()
sweep = run_sweep(
    window="hamming",
    filter_type="lowpass",
    orders=orders_lp,
    cutoffs=cutoffs,
    Nitera=params["Nitera"],
    Nmap=params["Nmap"],
    n_initial=params["n_initial"],
    seed=42,
)
print(f"Time: {time.time() - t0:.1f}s")
chaotic = np.sum(np.isfinite(sweep.h) & (sweep.h > 0))
periodic = np.sum(np.isfinite(sweep.h) & (sweep.h <= 0))
divergent = np.sum(~np.isfinite(sweep.h))
total = sweep.h.size
print(f"Chaotic: {chaotic}/{total} ({100 * chaotic / total:.1f}%)")
print(f"Periodic: {periodic}/{total} ({100 * periodic / total:.1f}%)")
print(f"Divergent: {divergent}/{total} ({100 * divergent / total:.1f}%)")

## 6. Visualização

Mapa de calor + classificação interleaved.

---

## 6. Visualisation

Continuous heatmap + interleaved classification.


In [ ]:
plot_heatmap_continuous(sweep)
plt.show()
plot_classification_interleaved(sweep)
plt.show()

## 7. Próximos passos

- **Sweeps completos**: `chaotic-pfc run sweep compute --window hamming --filter lowpass`
- **Varredura β (Kaiser)**: `chaotic-pfc run sweep beta-sweep --beta-min 2 --beta-max 14 --beta-step 0.5`
- **Análise estatística**: `chaotic-pfc run analysis`
- **Tabelas LaTeX para o PFC**: `chaotic-pfc run analysis export-tables`
- **Validação científica**: `docs/validation.rst` (erros menores que 0,35% em relação a Wolf (1985) e Sprott (2003))

---

## 7. Next steps

- **Full sweeps**: `chaotic-pfc run sweep compute --window hamming --filter lowpass`
- **β sweep (Kaiser)**: `chaotic-pfc run sweep beta-sweep --beta-min 2 --beta-max 14 --beta-step 0.5`
- **Statistical analysis**: `chaotic-pfc run analysis`
- **LaTeX tables for the thesis**: `chaotic-pfc run analysis export-tables`
- **Scientific validation**: `docs/validation.rst` (errors below 0.35% against Wolf (1985) and Sprott (2003))
